# Analysts requirements

This notebook contains data queries (required by analysts) along with the explanation.

In [ ]:
%matplotlib inline

import sqlite3
import polars as pl
import matplotlib.pyplot as plt

In [ ]:
conn = sqlite3.connect('db.sqlite')

# 1) 

> **You are working with an analyst that would like to be able to graph the population of any major metropolitan area in the US over time. Annual estimates are sufficient for this customer**

First, lets query the relevant data from the `population` table in our database. We will use polars library to handle the SQL query and DB interface.

Notice, we are concerned about the metropolitan areas only. In the `population` table, the column `STCOU` (State and county code) represents the county code. When this column is NULL, the row corresponds to a metropolitan area not a county.

In [ ]:
sql1 = """
SELECT 
    NAME, YEAR, POPULATION_EST
FROM
    population
WHERE
    STCOU is NULL;
"""

In [ ]:
df1 = pl.read_database(sql1, conn)

Lets graph the popluation of some random sample of metropolitan areas in the US over the past 8 years

In [ ]:
from random import choice

metro_names = df1['NAME'].to_list()
sample_metro_names = [choice(metro_names) for _ in range(3)]
                      
for metro in sample_metro_names:
    subset = df1.filter(pl.col('NAME') == metro)
    plt.figure()
    plt.plot(subset['YEAR'].to_list(), subset['POPULATION_EST'].to_list())
    plt.title(f'Population of: {metro}')
    plt.show()

<hr>

# 2) 

> **A different analyst wants to know about population and unemployment rates of the US at the county level. Annual estimates are sufficient for this customer**

For this requirement, we will need data from two tables in our database `population` and `unemployment`, where they will be joined on the county code and the year.

In [ ]:
sql2 = """
SELECT 
    unemployment.Area_name as county,
    population.year,
    population.POPULATION_EST as population,
    unemployment.unemployment_rate
FROM
    unemployment
JOIN 
    population 
ON 
    population.STCOU = unemployment.FIPStxt 
AND
    population.YEAR = unemployment.Year
"""

In [ ]:
df2 = pl.read_database(sql2, conn)

Here is an example of the queried data:

In [ ]:
df2.head(4)

Notice, the above dataframe is not analysis-friendly!

So we can simplify it into a pivot table where the index is a joint of the county name and year. This way it is easier to anlayze population vs. unemployment rate for a single county over time.

In [ ]:
index_cols = df2.columns[:-2]
value_cols = df2.columns[-2:]
df2 = df2.group_by(index_cols).agg([pl.col(c).mean() for c in value_cols]).sort(index_cols)

In [ ]:
df2.write_csv('pop_unemp_county_US.csv') # optionally save to a csv file

In [ ]:
df2